# 01 Data Prep
Loads NHAMCS ED SAS files, performs core cleaning, and writes model-ready CSVs to data/.

Outputs:
- data/eda_df.csv
- data/model_A_arrival_dataset.csv
- data/model_B_retrospective_dataset.csv

In [1]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

data_folder = Path('../data')
sas_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.sas7bdat')])
print(f'Found {len(sas_files)} SAS files')
for f in sas_files:
    print('-', f)

dfs = []
for file in sas_files:
    file_path = data_folder / file
    tmp = pd.read_sas(file_path, format='sas7bdat')
    if 'YEAR' not in tmp.columns:
        match = re.search(r'(20\d{2}|15|16|17|18)$', file.replace('.sas7bdat', '').replace('_sas', ''))
        if match:
            y = int(match.group(1))
            if y < 100:
                y = 2000 + y
            tmp['YEAR'] = y
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True, sort=False)
print('Combined shape:', df.shape)

Found 6 SAS files
- ed2015-sas.sas7bdat
- ed2016_sas.sas7bdat
- ed2017_sas.sas7bdat
- ed2018_sas.sas7bdat
- ed2021_sas.sas7bdat
- ed2022_sas.sas7bdat
Combined shape: (109760, 1061)


In [2]:
candidate_cols = [
    'WAITTIME', 'ARRTIME', 'VMONTH', 'VDAYR', 'YEAR',
    'AGE', 'SEX', 'RACEUN', 'ETHUN', 'IMMEDR', 'PAINSCALE',
    'LOV', 'ADMIT', 'FASTTRAK', 'OBSCLIN', 'BOARD', 'BOARDED', 'BOARDHOS',
    'BPSYS', 'BPDIAS', 'TEMPF', 'PULSE', 'RESPR', 'POPCT',
    'BEDREG', 'IMBED', 'BEDCZAR', 'MRI', 'XRAY', 'CT', 'CTCONTRAST',
    'ULTRASOUND', 'ANYIMAGE', 'LABTEST', 'TOTPROC', 'PATWT', 'EDWT', 'SETTYPE'
]
existing_cols = [c for c in candidate_cols if c in df.columns]
eda_df = df[existing_cols].copy()

def convert_arrtime_to_hour(arr_time):
    try:
        if pd.isna(arr_time):
            return np.nan
        s = arr_time.decode('utf-8') if isinstance(arr_time, bytes) else str(arr_time)
        s = ''.join(ch for ch in s if ch.isdigit())
        if len(s) == 3:
            s = '0' + s
        if len(s) != 4:
            return np.nan
        hh = int(s[:2])
        mm = int(s[2:])
        if 0 <= hh <= 23 and 0 <= mm <= 59:
            return hh
        return np.nan
    except Exception:
        return np.nan

if 'ARRTIME' in eda_df.columns:
    eda_df['ARRIVAL_HOUR'] = eda_df['ARRTIME'].apply(convert_arrtime_to_hour)

if 'WAITTIME' in eda_df.columns:
    eda_df['WAITTIME'] = pd.to_numeric(eda_df['WAITTIME'], errors='coerce')
    eda_df = eda_df[eda_df['WAITTIME'].between(0, 480)]

for c in ['YEAR', 'VMONTH', 'VDAYR', 'AGE', 'PAINSCALE', 'BPSYS', 'BPDIAS', 'TEMPF', 'PULSE', 'RESPR', 'POPCT']:
    if c in eda_df.columns:
        eda_df[c] = pd.to_numeric(eda_df[c], errors='coerce')

if 'VMONTH' in eda_df.columns:
    eda_df.loc[~eda_df['VMONTH'].between(1, 12), 'VMONTH'] = np.nan
if 'VDAYR' in eda_df.columns:
    eda_df.loc[~eda_df['VDAYR'].between(1, 7), 'VDAYR'] = np.nan
if 'ARRIVAL_HOUR' in eda_df.columns:
    eda_df.loc[~eda_df['ARRIVAL_HOUR'].between(0, 23), 'ARRIVAL_HOUR'] = np.nan

eda_df = eda_df.drop_duplicates()
print('EDA shape:', eda_df.shape)

EDA shape: (91812, 36)


In [3]:
arrival_cols = [c for c in [
    'WAITTIME', 'YEAR', 'ARRTIME', 'ARRIVAL_HOUR', 'VMONTH', 'VDAYR',
    'AGE', 'SEX', 'RACEUN', 'ETHUN', 'IMMEDR', 'PAINSCALE',
    'BPSYS', 'BPDIAS', 'TEMPF', 'PULSE', 'RESPR', 'POPCT'
] if c in eda_df.columns]

retro_cols = [c for c in [
    'WAITTIME', 'YEAR', 'ARRTIME', 'ARRIVAL_HOUR', 'VMONTH', 'VDAYR',
    'AGE', 'SEX', 'RACEUN', 'ETHUN', 'IMMEDR', 'PAINSCALE',
    'BPSYS', 'BPDIAS', 'TEMPF', 'PULSE', 'RESPR', 'POPCT',
    'LOV', 'ADMIT', 'FASTTRAK', 'OBSCLIN', 'BOARD', 'BOARDED', 'BOARDHOS',
    'BEDREG', 'IMBED', 'BEDCZAR', 'MRI', 'XRAY', 'CT', 'CTCONTRAST',
    'ULTRASOUND', 'ANYIMAGE', 'LABTEST', 'TOTPROC'
] if c in eda_df.columns]

model_A_df = eda_df[arrival_cols].copy().reset_index(drop=True)
model_B_df = eda_df[retro_cols].copy().reset_index(drop=True)

out = Path('../data')
out.mkdir(parents=True, exist_ok=True)
eda_df.to_csv(out / 'eda_df.csv', index=False)
model_A_df.to_csv(out / 'model_A_arrival_dataset.csv', index=False)
model_B_df.to_csv(out / 'model_B_retrospective_dataset.csv', index=False)

print('Saved:', out / 'eda_df.csv')
print('Saved:', out / 'model_A_arrival_dataset.csv')
print('Saved:', out / 'model_B_retrospective_dataset.csv')

if 'YEAR' in model_A_df.columns:
    print('Year counts (Model A):')
    display(model_A_df.groupby('YEAR', dropna=False).size().reset_index(name='rows').sort_values('YEAR'))

Saved: ../data/eda_df.csv
Saved: ../data/model_A_arrival_dataset.csv
Saved: ../data/model_B_retrospective_dataset.csv
Year counts (Model A):


,YEAR,rows
0,2015.0,17035
1,2016.0,16243
2,2017.0,14327
3,2018.0,17161
4,2021.0,13830
5,2022.0,13216
